<a href="https://colab.research.google.com/github/fauzanrahmatullah/information-retrieval/blob/main/crawling-information-retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!git config --global user.email "fauzan1464@gmail.com"
!git config --global user.name "fauzanrahmatullah"

In [7]:
import requests
import time
import re
from urllib.robotparser import RobotFileParser
from bs4 import BeautifulSoup
from collections import defaultdict

#1 portal berita "upnjatim.ac.id"
seed_url = "https://upnjatim.ac.id"
rp = RobotFileParser()
rp.set_url(f"{seed_url}/robots.txt")
try:
    rp.read()
    print("sip berhasil membaca robots.txt")
except:
    print("kok gagal membaca robots.txt")

Berhasil membaca robots.txt


In [8]:
#2 crawler
user_agent = "Fauzan_Informatics_Crawler_Bot/1.0"
frontier = [seed_url]
visited = set()
documents = {}
batas_maksimal = 200
graf_tautan = defaultdict(list) #untuk ranking di nomor 6
doc_id = 0

In [9]:
#3 ekstrak isi
def ekstrak_isi(html, base_url):
    soup = BeautifulSoup(html, 'html.parser')

    # pembersihan boilerplate
    for element in soup(["script", "style", "nav", "footer", "header", "aside"]):
        element.decompose()

    title = soup.title.string if soup.title else "Tanpa Judul"
    title = title.strip()

    tanggal = "Tidak ditemukan"
    time_tag = soup.find('time')
    if time_tag:
        tanggal = time_tag.get_text(strip=True)

    paragraphs = soup.find_all('p')
    isi_utama = " ".join([p.get_text(strip=True) for p in paragraphs])

    links = []
    for a_tag in soup.find_all('a', href=True):
        href = a_tag['href']
        if href.startswith('/'):
            href = base_url + href
        if href.startswith(base_url):
            href = href.split('#')[0].split('?')[0] # hilangkan (# dan ?) pada link
            if href not in links:
                links.append(href)
    return title, tanggal, isi_utama, links

print(f"starting crawling {batas_maksimal} pages")

# jalankan crawler
while frontier and doc_id < batas_maksimal:
    url = frontier.pop(0)
    if url in visited: continue

    # robots.txt permission
    if not rp.can_fetch(user_agent, url):
        visited.add(url)
        continue

    print(f"[{doc_id+1}/{batas_maksimal}] Mengunduh: {url}")
    try:
        headers = {'User-Agent': user_agent}
        response = requests.get(url, headers=headers, timeout=10)
        visited.add(url)

        if response.status_code == 200 and 'text/html' in response.headers.get('Content-Type', ''):
            # fungsi ekstraksi Tugas 3
            judul, tanggal, isi, tautan_baru = ekstrak_isi(response.text, seed_url)

            if isi:
                documents[doc_id] = {'url': url, 'judul': judul, 'tanggal': tanggal, 'isi': isi}
                graf_tautan[url] = tautan_baru
                doc_id += 1

                # masukan link baru ke antrean
                for t in tautan_baru:
                    if t not in visited and t not in frontier:
                        frontier.append(t)

        # two tick timer
        time.sleep(2)
    except Exception as e:
        visited.add(url)

print("Selesai mengunduh!")

starting crawling 200 pages
[1/200] Mengunduh: https://upnjatim.ac.id
[2/200] Mengunduh: https://upnjatim.ac.id/
[3/200] Mengunduh: https://upnjatim.ac.id/sejarah/
[4/200] Mengunduh: https://upnjatim.ac.id/visi-misi/
[5/200] Mengunduh: https://upnjatim.ac.id/struktur-organisasi-2/
[5/200] Mengunduh: https://upnjatim.ac.id/kampus-bela-negara/
[6/200] Mengunduh: https://upnjatim.ac.id/sasaran-strategis/
[7/200] Mengunduh: https://upnjatim.ac.id/tujuan/
[8/200] Mengunduh: https://upnjatim.ac.id/hymne-mars/
[9/200] Mengunduh: https://upnjatim.ac.id/lambang-logo/
[9/200] Mengunduh: https://upnjatim.ac.id/tarif-layanan/
[10/200] Mengunduh: https://upnjatim.ac.id/seleksi-masuk/
[11/200] Mengunduh: https://upnjatim.ac.id/akreditasi-instansi-prodi/
[12/200] Mengunduh: https://upnjatim.ac.id/kalender-akademik-ta-2025-2026/
[12/200] Mengunduh: https://upnjatim.ac.id/program-sarjana/
[13/200] Mengunduh: https://upnjatim.ac.id/program-pascasarjana/
[14/200] Mengunduh: https://upnjatim.ac.id/categor

In [30]:
#4 inverted index
inverted_index = defaultdict(set)

for d_id, data in documents.items():
    teks_mentah = data['judul'] + " " + data['isi']
    teks_bersih = re.sub(r'[^a-z0-9\s]', '', teks_mentah.lower())
    tokens = teks_bersih.split()

    for token in tokens:
        inverted_index[token].add(d_id)

print(f"dapat {len(documents)} dokumen.")
print(f"kosakata unik: {len(inverted_index)}")

# testing
kata_target = 'rektor'
if kata_target in inverted_index:
    doc_ids = list(inverted_index[kata_target])
    print(f"\nKata '{kata_target}' ada di {len(doc_ids)} dokumen.")
    print(f"DocID: {doc_ids}")
else:
    print(f"\nKata '{kata_target}' tidak ditemukan.")

dapat 116 dokumen.
kosakata unik: 5538

Kata 'rektor' ada di 41 dokumen.
DocID: [8, 20, 24, 46, 47, 51, 53, 55, 60, 66, 67, 68, 71, 72, 73, 74, 75, 76, 77, 81, 82, 83, 85, 87, 90, 92, 93, 98, 99, 101, 102, 103, 104, 105, 108, 109, 110, 111, 112, 113, 114]


In [23]:
#5 deteksi near duplicate
def get_shingles(text, k=3):
    teks_bersih = re.sub(r'[^a-z0-9\s]', '', text.lower())
    tokens = teks_bersih.split()
    shingles = set()
    for i in range(len(tokens) - k + 1):
        shingles.add(" ".join(tokens[i:i+k]))
    return shingles

def jaccard_sim(set1, set2):
    if not set1 or not set2: return 0.0
    return len(set1.intersection(set2)) / len(set1.union(set2))

print("membuat shingle dan membandingkan dokumen (Batas Jaccard > 0.8)")
doc_shingles = {d_id: get_shingles(data['isi'], k=3) for d_id, data in documents.items()}
near_dupes = []
ids = list(documents.keys())

for i in range(len(ids)):
    for j in range(i + 1, len(ids)):
        id1, id2 = ids[i], ids[j]
        sim = jaccard_sim(doc_shingles[id1], doc_shingles[id2])
        if sim > 0.8:
            near_dupes.append((id1, id2, sim))

if near_dupes:
    print(f"\nketemu {len(near_dupes)} buah duplikat:")
    for id1, id2, sim in near_dupes:
        print(f" - Doc {id1} & Doc {id2} (Kemiripan: {sim:.3f})")
else:
    print("\nndak ada artikel yang tingkat kemiripannya di atas 0.8.")

membuat shingle dan membandingkan dokumen (Batas Jaccard > 0.8)

ketemu 3 buah duplikat:
 - Doc 0 & Doc 1 (Kemiripan: 1.000)
 - Doc 38 & Doc 54 (Kemiripan: 1.000)
 - Doc 43 & Doc 58 (Kemiripan: 1.000)


In [31]:
#6 hitung page rank link
url_to_id = {data['url']: d_id for d_id, data in documents.items()}
valid_urls = set(url_to_id.keys())
adj_list = defaultdict(list)
out_deg = defaultdict(int)

# penentuan arah link
for src, targets in graf_tautan.items():
    if src in valid_urls:
        src_id = url_to_id[src]
        for tgt in set(targets):
            if tgt in valid_urls and tgt != src:
                adj_list[src_id].append(url_to_id[tgt])
                out_deg[src_id] += 1

# nilai awal page rank
N = len(valid_urls)
d = 0.85
pr = {d_id: 1/N for d_id in url_to_id.values()}

# iterasi
print("hitungan page rank")
for _ in range(30):
    new_pr = {}
    for node in url_to_id.values():
        sum_in = sum(pr[u] / out_deg[u] for u, v_list in adj_list.items() if node in v_list)
        new_pr[node] = ((1 - d) / N) + (d * sum_in)
    pr = new_pr

# print hasil teratas 10 biji
ranked = sorted(pr.items(), key=lambda x: x[1], reverse=True)
print("\ntop 10:")
for rank, (d_id, score) in enumerate(ranked[:10], 1):
    print(f"{rank}. [Doc {d_id}] URL: {documents[d_id]['url']} (Skor: {score:.5f})")

pembangunan grafik ranking
hitungan page rank

top 10:
1. [Doc 16] URL: https://upnjatim.ac.id/vtour/ (Skor: 0.03518)
2. [Doc 17] URL: https://upnjatim.ac.id/unit-kegiatan-mahasiswa/ (Skor: 0.03433)
3. [Doc 1] URL: https://upnjatim.ac.id/ (Skor: 0.03407)
4. [Doc 2] URL: https://upnjatim.ac.id/sejarah/ (Skor: 0.03360)
5. [Doc 3] URL: https://upnjatim.ac.id/visi-misi/ (Skor: 0.03360)
6. [Doc 4] URL: https://upnjatim.ac.id/kampus-bela-negara/ (Skor: 0.03360)
7. [Doc 5] URL: https://upnjatim.ac.id/sasaran-strategis/ (Skor: 0.03360)
8. [Doc 6] URL: https://upnjatim.ac.id/tujuan/ (Skor: 0.03360)
9. [Doc 7] URL: https://upnjatim.ac.id/hymne-mars/ (Skor: 0.03360)
10. [Doc 8] URL: https://upnjatim.ac.id/tarif-layanan/ (Skor: 0.03360)
